In [ ]:
%load_ext autoreload
%autoreload 2
import os

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")

In [ ]:
# Read JSONL directly into a DataFrame
df = pd.read_json("data/traces/large_dataset.jsonl", lines=True)
print(f"DataFrame shape: {df.shape}")
print(f"DataFrame columns: {df.columns.tolist()}")
print(f"DataFrame head:\n{df.head()}")

In [ ]:
solvers = list(df["traces"][0].keys())
lookup = {solver: torch.tensor(i, dtype=torch.long) for i, solver in enumerate(solvers)}
X_list, y_list = [], []
for _, row in df.iterrows():
    for trace_name, trace in row["traces"].items():
        X_list.append(torch.tensor(trace["1.0"][0]["steps"], dtype=torch.float32))
        y_list.append(lookup[trace_name])
print(f"Number of samples: {len(X_list)}")
X = torch.nn.utils.rnn.pad_sequence(X_list, batch_first=True).flatten(start_dim=1)
y = torch.stack(y_list)
print(X.shape, y.shape)

In [ ]:
print(lookup)

In [ ]:
X_train = X[: int(0.8 * len(X))]
y_train = y[: int(0.8 * len(y))]
X_test = X[int(0.8 * len(X)) :]
y_test = y[int(0.8 * len(y)) :]
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# 1. Model: linear probe on top of input features
# ---------------------------------------------------------
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)  # logits


class MLPProbe(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)  # logits


def train_probe(model, train_loader, device, epochs=20, lr=1e-3, weight_decay=1e-4):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(epochs):
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            n += x.size(0)

        print(f"Epoch {epoch + 1}: loss={total_loss / n:.4f} acc={correct / n:.4f}")

    return model


# ---------------------------------------------------------
# 2. Confusion-matrix similarity (behavioral)
# ---------------------------------------------------------
@torch.no_grad()
def confusion_similarity(model, loader, num_classes, device):
    model.eval()
    conf = torch.zeros(
        num_classes, num_classes, device=device
    )  # rows: true, cols: predicted prob mass
    counts = torch.zeros(num_classes, device=device)

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        probs = F.softmax(model(x), dim=1)
        for c in range(num_classes):
            mask = y == c
            if mask.any():
                conf[c] += probs[mask].sum(dim=0)
                counts[c] += mask.sum()

    conf = conf / counts.clamp(min=1).unsqueeze(1)  # avg predicted prob distribution per true class

    # symmetrize -> similarity matrix
    sim = (conf + conf.T) / 2
    return sim.cpu()  # sim[i, j] high => i and j are often confused


# ---------------------------------------------------------
# 3. Activation-centroid similarity (representational)
# ---------------------------------------------------------
@torch.no_grad()
def activation_centroid_similarity(model, loader, num_classes, device, use_logits=True):
    """
    Computes per-class centroids and cosine similarity between them, after
    centering by the global mean. Centering matters: without it, cosine
    similarity is dominated by whatever structure is shared across all
    classes (e.g. a large common offset), which crushes every pairwise
    similarity toward 1.0 regardless of real class differences.

    use_logits=True uses the trained probe's output logits as the
    representation (a natural choice since you already have them).
    Set to False to use raw input features instead, or swap in a
    hidden-layer hook if using an MLP/backbone.
    """
    model.eval()
    sums = defaultdict(lambda: None)
    counts = defaultdict(int)
    global_sum = None
    global_count = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        feats = model(x) if use_logits else x  # swap in hook output here if needed

        # accumulate global mean
        s = feats.sum(dim=0)
        global_sum = s if global_sum is None else global_sum + s
        global_count += feats.size(0)

        # accumulate per-class sums
        for c in y.unique():
            c = c.item()
            mask = y == c
            cs = feats[mask].sum(dim=0)
            sums[c] = cs if sums[c] is None else sums[c] + cs
            counts[c] += mask.sum().item()

    global_mean = global_sum / global_count

    centroids = torch.stack(
        [
            sums[c] / counts[c] - global_mean  # center before comparing
            for c in range(num_classes)
        ]
    )

    centroids = F.normalize(centroids, dim=1)
    sim = centroids @ centroids.T
    return sim.cpu()

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt


def plot_similarity_matrix(sim_matrix, title, class_names=None):
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        sim_matrix.numpy(),
        annot=True,
        fmt=".2f",
        cmap="viridis",
        xticklabels=class_names or range(sim_matrix.shape[0]),
        yticklabels=class_names or range(sim_matrix.shape[0]),
    )
    plt.title(title)
    plt.show()

In [ ]:
# ---------------------------------------------------------
# 4. Usage
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(solvers)

linear_model = LinearProbe(input_dim=X_train.shape[1], num_classes=num_classes)
linear_model = train_probe(linear_model, train_loader, device, epochs=30, lr=1e-3)

conf_sim = confusion_similarity(linear_model, test_loader, num_classes=num_classes, device=device)
act_sim = activation_centroid_similarity(
    linear_model, test_loader, num_classes=num_classes, device=device
)
test_acc = (
    (linear_model(X_test.to(device)).argmax(dim=1) == y_test.to(device)).float().mean().item()
)
print(f"Test accuracy: {test_acc:.4f}")
plot_similarity_matrix(conf_sim, "Confusion-matrix similarity", class_names=solvers)
plot_similarity_matrix(act_sim, "Activation-centroid similarity", class_names=solvers)

In [ ]:
# ---------------------------------------------------------
# 4. Usage
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(solvers)

mlp_model = MLPProbe(input_dim=X_train.shape[1], hidden_dim=375, num_classes=num_classes)
mlp_model = train_probe(mlp_model, train_loader, device, epochs=30, lr=1e-3)

conf_sim = confusion_similarity(mlp_model, test_loader, num_classes=num_classes, device=device)
act_sim = activation_centroid_similarity(
    mlp_model, test_loader, num_classes=num_classes, device=device
)
test_acc = (mlp_model(X_test.to(device)).argmax(dim=1) == y_test.to(device)).float().mean().item()
print(f"Test accuracy: {test_acc:.4f}")
plot_similarity_matrix(conf_sim, "Confusion-matrix similarity", class_names=solvers)
plot_similarity_matrix(act_sim, "Activation-centroid similarity", class_names=solvers)

In [ ]:


def plot_weights(model, class_names=None):
    W = model.fc.weight.detach().cpu()  # [num_classes, input_dim]

    plt.figure(figsize=(10, 4))
    sns.heatmap(W, cmap="coolwarm", center=0, yticklabels=class_names or range(W.shape[0]))
    plt.xlabel("Input feature dim")
    plt.ylabel("Class")
    plt.title("Linear probe weights per class")
    plt.show()

    # cosine similarity between weight rows (this is your idea #2 from earlier)
    Wn = F.normalize(W, dim=1)
    weight_sim = Wn @ Wn.T
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        weight_sim,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        xticklabels=class_names or range(W.shape[0]),
        yticklabels=class_names or range(W.shape[0]),
    )
    plt.title("Cosine similarity between class weight vectors")
    plt.show()


plot_weights(linear_model, class_names=solvers)

In [ ]:
import torch


def plot_weights_as_grids(model, class_idx, seq_len=15, grid_size=5, class_names=None):
    """
    Plots the weight vector for one class, reshaped back into its original
    (seq_len, grid_size, grid_size) structure -- one small heatmap per
    sequence step, arranged in a row.
    """
    W = model.fc.weight.detach().cpu()  # [num_classes, 375]
    w = W[class_idx].reshape(seq_len, grid_size, grid_size)  # [15, 5, 5]

    vmax = w.abs().max()  # shared color scale across all 15 grids, centered at 0

    fig, axes = plt.subplots(1, seq_len, figsize=(seq_len * 1.5, 2))
    for t in range(seq_len):
        ax = axes[t]
        im = ax.imshow(w[t], cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(f"t={t}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    name = class_names[class_idx] if class_names else class_idx
    fig.suptitle(f"Class {name} — weights per timestep")
    fig.colorbar(im, ax=axes, shrink=0.6, label="weight")
    plt.show()


def plot_all_class_weights(model, num_classes, seq_len=15, grid_size=5, class_names=None):
    """Loops over every class and plots its weight grids."""
    for c in range(num_classes):
        plot_weights_as_grids(model, c, seq_len, grid_size, class_names)


plot_all_class_weights(linear_model, num_classes, seq_len=15, grid_size=5, class_names=solvers)